# Evaluacion de modelo: BAAI/bge-m3

**Estado del arte multilingue (lento, 1024 dim, max precision)**

- Dimension: 1024
- Batch size GPU: 16
- Vector store: `./vector_store_bge_m3/`
- Coleccion: `corpus_upeu_bge_m3`


## 1. Setup, GPU y rutas


In [17]:
import os
import re
import time
import json
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# --- Deteccion de GPU ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()
else:
    print("  AVISO: No se detecto GPU. El rendimiento sera menor.")

# --- Rutas ---
TXT_LIMPIO      = Path("/home/jupyteruser/work/corpus_upeu/txt_limpio")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")
VECTOR_STORE    = Path(f"/home/jupyteruser/work/vector_store_bge_m3")
COLLECTION_NAME = f"corpus_upeu_bge_m3"

os.makedirs(VECTOR_STORE, exist_ok=True)
os.makedirs(METADATA_FOLDER, exist_ok=True)

print(f"\nVector store: {VECTOR_STORE}")
print(f"Coleccion:    {COLLECTION_NAME}")


Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
  VRAM total: 6.4 GB

Vector store: /home/jupyteruser/work/vector_store_bge_m3
Coleccion:    corpus_upeu_bge_m3


## 2. Cargar modelo `BAAI/bge-m3`


In [18]:
print("Cargando modelo {'BAAI/bge-m3'}...")
t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device=device)
MODEL_NAME = "BAAI/bge-m3"
print(f"  Cargado en {time.time()-t0:.1f}s")
print(f"  Dimension embedding: {model.get_sentence_embedding_dimension()}")
print(f"  Max seq length:      {model.max_seq_length}")


Cargando modelo {'BAAI/bge-m3'}...


/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  Cargado en 32.6s
  Dimension embedding: 1024
  Max seq length:      8192


## 3. Cargar textos limpios (del notebook 2)


In [19]:
txt_files = sorted(TXT_LIMPIO.glob("*.txt"))
print(f"Documentos disponibles: {len(txt_files)}")
for f in txt_files[:5]:
    print(f"  {f.name}")
print(f"  ... y {max(0, len(txt_files)-5)} mas")


Documentos disponibles: 48
  DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt
  ESTATUTO 2024. 04-09-2024.txt
  Guía para la organización y orientación del legajo para la docencia ordinaria.txt
  MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt
  Modelo de índice del contenido - Legajo.txt
  ... y 43 mas


## 4. Chunking estructural (300 tokens, overlap 80)


In [20]:
_RE_BLOQUE = re.compile(
    r'(?:^|\n)\s*('
    r'Art[íi]culo\s+\d+[ºo°]?(?:\s*[.-]\s*\d+)?'
    r'|Cap[íi]tulo\s+[IVXLCDM\d]+'
    r'|Secci[óo]n\s+\d+'
    r'|T[íi]tulo\s+[IVXLCDM\d]+'
    r')',
    re.IGNORECASE
)


def chunk_text_estructural(text, tokenizer, max_tokens=300, overlap=80, max_chars_per_chunk=1500):
    matches = list(_RE_BLOQUE.finditer(text))
    if not matches:
        return chunk_text_simple(text, tokenizer, max_tokens, overlap, max_chars_per_chunk)
    chunks = []
    buffer_cabecera = None
    buffer_contenido = []
    for i, m in enumerate(matches):
        cabecera = m.group(1).strip()
        inicio_contenido = m.end()
        fin_contenido = matches[i+1].start() if i+1 < len(matches) else len(text)
        contenido = text[inicio_contenido:fin_contenido].strip()
        bloque = f"{cabecera}\n{contenido}" if contenido else cabecera
        tokens = tokenizer.encode(bloque, add_special_tokens=False)
        if len(tokens) <= max_tokens:
            if buffer_cabecera is None:
                buffer_cabecera = cabecera
                buffer_contenido = [contenido] if contenido else []
            else:
                buffer_contenido.append(contenido)
                buffer_completo = f"{buffer_cabecera}\n" + "\n".join(buffer_contenido)
                if len(tokenizer.encode(buffer_completo, add_special_tokens=False)) > max_tokens:
                    chunks.append(buffer_completo)
                    buffer_cabecera = cabecera
                    buffer_contenido = [contenido] if contenido else []
        else:
            if buffer_cabecera is not None:
                chunks.append(f"{buffer_cabecera}\n" + "\n".join(buffer_contenido))
                buffer_cabecera = None
                buffer_contenido = []
            for j in range(0, len(tokens), max_tokens - overlap):
                sub = tokenizer.decode(tokens[j:j+max_tokens], skip_special_tokens=True)
                sub = sub[:max_chars_per_chunk]
                chunks.append(f"{cabecera}\n{sub}")
    if buffer_cabecera is not None:
        chunks.append(f"{buffer_cabecera}\n" + "\n".join(buffer_contenido))
    return [c for c in chunks if c.strip()]


def chunk_text_simple(text, tokenizer, max_tokens=300, overlap=80, max_chars_per_chunk=1500):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk = tokenizer.decode(tokens[start:end], skip_special_tokens=True)
        if len(chunk) > max_chars_per_chunk:
            chunk = chunk[:max_chars_per_chunk]
        chunks.append(chunk)
        start += (max_tokens - overlap)
    return chunks


# --- Generar chunks de todos los documentos ---
tokenizer = model.tokenizer
all_chunks = []
for txt_path in txt_files:
    doc_name = txt_path.stem
    text = txt_path.read_text(encoding='utf-8')
    chunks = chunk_text_estructural(text, tokenizer, max_tokens=300, overlap=80)
    for i, ch in enumerate(chunks):
        all_chunks.append({
            "documento": doc_name,
            "chunk_id": f"{doc_name}_{i:04d}",
            "texto": ch,
        })
    print(f"  {doc_name[:50]:50} -> {len(chunks):4} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")


  DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025     ->   27 chunks


Token indices sequence length is longer than the specified maximum sequence length for this model (8568 > 8192). Running this sequence through the model will result in indexing errors


  ESTATUTO 2024. 04-09-2024                          ->  113 chunks
  Guía para la organización y orientación del legajo ->   39 chunks
  MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER A ->   16 chunks
  Modelo de índice del contenido - Legajo            ->    6 chunks
  Politica Institucional de Inclusión y diversidad c ->    4 chunks
  Politica Institucional de trabajo digno y protecci ->    4 chunks
  Politica-ambiental                                 ->    3 chunks
  REGLAMENTO ADMISION 2025.v7                        ->  111 chunks
  REGLAMENTO BECAS 2021 ACTUALIZADO                  ->   74 chunks
  REGLAMENTO CODIGO ETICA INVESTIGACION 2021         ->   21 chunks
  REGLAMENTO DE ESTUDIOS POSGRADO 2025               ->  321 chunks
  REGLAMENTO DE ESTUDIOS V5_2025                     ->  387 chunks
  REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4        ->   43 chunks
  REGLAMENTO DOCENCIA ORDINARIA v3.5                 ->   79 chunks
  REGLAMENTO ESTUDIANTE UNIONISTA V3            

## 5. Generar embeddings en GPU


In [21]:
# BGE-m3 requiere prefijo en pasajes para indexar
textos = ['passage: ' + c['texto'] for c in all_chunks]
print(f"Prefijando {len(textos)} chunks con 'passage: '")



Prefijando 4152 chunks con 'passage: '


In [22]:
batch = 16
print(f"\nEncoding {len(textos)} chunks en {device} con batch={batch}...")
t0 = time.time()
embeddings = model.encode(
    textos,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
dt = time.time() - t0
print(f"\nEncoding completo en {dt:.1f}s ({len(embeddings)/dt:.0f} chunks/s)")
print(f"Shape: {embeddings.shape}")
print(f"Memoria GPU usada: {torch.cuda.max_memory_allocated()/1e9:.2f} GB" if device=='cuda' else "")
torch.cuda.empty_cache() if device=='cuda' else None



Encoding 4152 chunks en cuda con batch=16...


Batches:   0%|          | 0/260 [00:00<?, ?it/s]


Encoding completo en 189.5s (22 chunks/s)
Shape: (4152, 1024)
Memoria GPU usada: 5.57 GB


## 6. Crear vector store y coleccion


In [23]:
client = chromadb.PersistentClient(
    path=str(VECTOR_STORE),
    settings=Settings(anonymized_telemetry=False, allow_reset=True)
)

# Borrar coleccion previa si existe
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Coleccion '{COLLECTION_NAME}' eliminada.")
except ValueError:
    print(f"No existia coleccion previa.")
except Exception as e:
    print(f"WARN: {type(e).__name__}: {e}")

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": "Corpus UPeU con modelo BAAI/bge-m3",
        "hnsw:space": "cosine"
    }
)
print(f"Coleccion creada. Listo para indexar.")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Coleccion 'corpus_upeu_bge_m3' eliminada.
Coleccion creada. Listo para indexar.


## 7. Indexar chunks con metadata enriquecida


In [24]:
CATEGORIAS_KEYWORDS = {
    "A": ["estatuto", "general upeu", "defensor", "comite electoral", "tupa",
          "reglamento interno de trabajo", "honores"],
    "B": ["estudios", "estudiante unionista", "docencia", "idiomas", "movilidad",
          "publicaciones y fondo", "pago servicios academicos", "becas", "admision",
          "credito", "matricula", "egresado"],
    "C": ["investigacion", "investigadores", "incentivos investigacion",
          "propiedad intelectual", "codigo etica investigacion", "etica"],
    "D": ["promocion", "recreacion", "deporte", "residencias", "multimedia",
          "seguimiento de egresados", "servicio psicologico"],
    "E": ["politica institucional", "politica-ambiental", "ambiental",
          "capacitacion docente", "auditoria interna", "seguridad y salud",
          "comedor", "transporte", "biblioteca"],
}

def obtener_categoria(doc_name):
    nombre = doc_name.lower()
    for cat, kws in CATEGORIAS_KEYWORDS.items():
        for kw in kws:
            if kw in nombre:
                return cat
    return "E"

_RE_ART = re.compile(
    r'(Art[íi]culo\s+\d+[ºo°]?(?:\s*[.-]\s*\d+)?|'
    r'Cap[íi]tulo\s+[IVXLCDM\d]+|'
    r'Secci[óo]n\s+\d+|'
    r'T[íi]tulo\s+[IVXLCDM\d]+)',
    re.IGNORECASE
)

def extraer_articulo(texto):
    m = _RE_ART.search(texto[:200])
    return m.group(1).strip() if m else ""

ids = [c["chunk_id"] for c in all_chunks]
metadatos = [{
    "documento": c["documento"],
    "categoria": obtener_categoria(c["documento"]),
    "articulo": extraer_articulo(c["texto"]),
    "chunk_id": c["chunk_id"],
    "num_chars": len(c["texto"]),
} for c in all_chunks]

# Indexar por lotes
BATCH = 500
print(f"Indexando {len(ids)} chunks en lotes de {BATCH}...")
for i in range(0, len(ids), BATCH):
    end = min(i + BATCH, len(ids))
    collection.add(
        ids=ids[i:end],
        documents=[c["texto"] for c in all_chunks[i:end]],
        metadatas=metadatos[i:end],
        embeddings=embeddings[i:end].tolist()
    )
print(f"Indexacion completa. Total: {collection.count()} chunks")


Indexando 4152 chunks en lotes de 500...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Indexacion completa. Total: 4152 chunks


## 8. Banco de 10 preguntas de prueba


In [25]:
PREGUNTAS = [
    ("¿Cuáles son los derechos del estudiante?", "B"),
    ("¿Cómo puedo reservar mi matrícula?", "B"),
    ("¿Cuál es el procedimiento para cambiar de carrera?", "B"),
    ("¿Qué sanciones existen en la universidad?", "A"),
    ("¿Cómo solicito una beca?", "B"),
    ("¿Cuál es el procedimiento para presentar una queja?", "A"),
    ("¿Qué dice el estatuto sobre el gobierno universitario?", "A"),
    ("¿Cómo se realiza un proyecto de investigación?", "C"),
    ("¿Qué servicios ofrece la universidad a los egresados?", "D"),
    ("¿Cuál es la política ambiental de la UPeU?", "E"),
]
print(f"Banco: {len(PREGUNTAS)} preguntas")


Banco: 10 preguntas


## 9. Evaluacion con 3 niveles de calidad


In [26]:
def encode_query(text):
    return model.encode(["query: " + text], normalize_embeddings=True)[0].tolist()


In [27]:
def evaluar_consulta(consulta, categoria_esperada=None, n_resultados=3,
                       umbral_excelente=0.30, umbral_aceptable=0.50):
    resultados = collection.query(
        query_embeddings=[encode_query(consulta)],
        n_results=n_resultados,
        include=["documents", "metadatas", "distances"]
    )
    metas = resultados["metadatas"][0] if resultados["metadatas"][0] else []
    dists = resultados["distances"][0] if resultados["distances"][0] else [1.0]
    top1_dist = dists[0]
    if top1_dist < umbral_excelente:
        calidad = "excelente"; cubierta = True
    elif top1_dist < umbral_aceptable:
        calidad = "aceptable"; cubierta = True
    else:
        calidad = "no_cubierta"; cubierta = False
    categoria_match = None
    if categoria_esperada and metas:
        categoria_match = (metas[0].get("categoria") == categoria_esperada)
    return {
        "consulta": consulta,
        "cubierta": cubierta,
        "calidad": calidad,
        "top1_dist": round(top1_dist, 4),
        "top1_doc": metas[0].get("documento", "?") if metas else "?",
        "top1_categoria": metas[0].get("categoria", "?") if metas else "?",
        "top1_articulo": metas[0].get("articulo", "") if metas else "",
        "categoria_esperada": categoria_esperada,
        "categoria_match": categoria_match,
    }


## 10. Ejecutar evaluacion completa


In [28]:
resultados = []
for pregunta, cat_esp in PREGUNTAS:
    r = evaluar_consulta(pregunta, categoria_esperada=cat_esp)
    resultados.append(r)
    e = "✓" if r["cubierta"] else "✗"
    cm = "✓" if r["categoria_match"] else ("✗" if r["categoria_match"] is False else "?")
    print(f"{e} [{r['calidad']:11}] d={r['top1_dist']:.3f} cat={r['top1_categoria']}{cm} {r['top1_articulo']:30} {pregunta[:50]}")

df_res = pd.DataFrame(resultados)
print(f"\n=== RESUMEN MODELO: {MODEL_NAME} ===")
print(f"Cubiertas (aceptable+excelente): {df_res['cubierta'].sum()}/{len(df_res)}")
print(f"Excelentes: {(df_res['calidad']=='excelente').sum()}/{len(df_res)}")
print(f"Categoria match: {df_res['categoria_match'].sum()}/{df_res['categoria_match'].notna().sum()}")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✓ [excelente  ] d=0.246 cat=A✗ Artículo 112°                  ¿Cuáles son los derechos del estudiante?
✓ [aceptable  ] d=0.334 cat=B✓ Artículo 65°                   ¿Cómo puedo reservar mi matrícula?
✓ [aceptable  ] d=0.358 cat=B✓ Artículo 282º                  ¿Cuál es el procedimiento para cambiar de carrera?
✓ [excelente  ] d=0.293 cat=C✗ Artículo 26°                   ¿Qué sanciones existen en la universidad?
✓ [aceptable  ] d=0.308 cat=A✗                                ¿Cómo solicito una beca?
✓ [aceptable  ] d=0.356 cat=A✓ Artículo 22°                   ¿Cuál es el procedimiento para presentar una queja
✓ [aceptable  ] d=0.351 cat=A✓ Artículo 20°                   ¿Qué dice el estatuto sobre el gobierno universita
✓ [aceptable  ] d=0.343 cat=C✓ Artículo 149°                  ¿Cómo se realiza un proyecto de investigación?
✓ [aceptable  ] d=0.381 cat=A✗ Artículo 870°                  ¿Qué servicios ofrece la universidad a los egresad
✓ [aceptable  ] d=0.362 cat=E✓                  

## 11. Prueba interactiva (estilo notebook 6)


In [29]:
def consultar(pregunta, n_resultados=3):
    r = collection.query(
        query_embeddings=[encode_query(pregunta)],
        n_results=n_resultados,
        include=["documents", "metadatas", "distances"]
    )
    print(f"\nConsulta: {pregunta}\n")
    for i, (doc, meta, dist) in enumerate(zip(
            r["documents"][0], r["metadatas"][0], r["distances"][0])):
        print(f"--- Resultado {i+1} (distancia: {dist:.4f}) ---")
        print(f"Documento: {meta.get('documento','?')}")
        print(f"Categoria: {meta.get('categoria','?')}")
        if meta.get('articulo'):
            print(f"Articulo:  {meta.get('articulo')}")
        print(f"Texto: {doc[:400]}...\n")

# --- Pregunta de prueba ---
consultar("¿Qué requisitos necesito para obtener una beca en la UPeU?")



Consulta: ¿Qué requisitos necesito para obtener una beca en la UPeU?

--- Resultado 1 (distancia: 0.2513) ---
Documento: REGLAMENTO BECAS 2021 ACTUALIZADO
Categoria: B
Articulo:  Artículo 49°
Texto: Artículo 49°
Artículo 49°. Para el otorgamiento de la beca, el estudiante presenta los siguientes requisitos: 49.1. Tener la calidad de estudiante regular de la UPeU. 49.2. Haber obtenido un promedio ponderado mínimo de catorce (14) y el cien 49.3. No haber sido ni estar sancionado en el semestre académico precedente ni 49.4. No adeudar a la UPeU por ningún concepto. 49.5. Mostrar los documentos d...

--- Resultado 2 (distancia: 0.2523) ---
Documento: REGLAMENTO PRODAC 2022(R_SG)
Categoria: E
Articulo:  Artículo 22°
Texto: Artículo 22°
. Requisitos generales. Son requisitos generales para postular a la obtención de la beca o beneficio:

22.1. Inscripción. Inscribirse en el sitio web de la UPeU.

22.2. Presentación documentaria. El postulante debe adjuntar, sucesiva, simultánea y obligatori

## 12. Widget interactivo (escribe tu propia consulta)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

txt_input = widgets.Text(
    value='¿Cómo puedo cambiar de carrera?',
    placeholder='Escribe tu consulta...',
    description='Consulta:',
    layout=widgets.Layout(width='80%')
)
btn = widgets.Button(description='Consultar', button_style='primary')
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        consultar(txt_input.value, n_resultados=3)

btn.on_click(on_click)
display(widgets.VBox([txt_input, btn, out]))
